# In-memory HEFT operators

Run a small native pmwd simulation and advect five hybrid effective field theory (HEFT) operators, starting from white-noise modes. All intermediate arrays stay in memory.

From a checkout containing `pmwd/heft.py`, install pmwd and the example dependencies in a development environment with `python -m pip install -e '.[vis]' jupyterlab`. Open this notebook in that environment and run all cells in order in a **fresh kernel**. As in `quickstart.ipynb`, leave device selection to JAX: an available GPU can be used when JAX is installed with GPU support, with CPU execution also supported. Select GPU visibility in your environment before launching the kernel. The first cell prints the backend and visible devices. This example uses double precision; its small grid and coarse time steps demonstrate the interface, not production accuracy.


In [ ]:
import os
# Uncomment to explicitly select CPU execution before JAX initializes a backend.
# os.environ['JAX_PLATFORMS'] = 'cpu'

import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

from pmwd import Configuration, Cosmology, white_noise
from pmwd.heft import simulate_heft

print('JAX backend:', jax.default_backend())
print('JAX devices:', jax.devices())


## Configuration and initial conditions

`Configuration` fixes the periodic box, particle/force grids, time steps, and numerical precision. Here the box is $64\,h^{-1}\mathrm{Mpc}$, with $8^3$ particles, a $16^3$ force mesh, and an $8^3$ HEFT output grid. Particle and HEFT arrays use `float64`; float32 CIC accumulation can limit cancellation-sensitive statistics.


In [ ]:
conf = Configuration(
    ptcl_spacing=8., ptcl_grid_shape=(8,) * 3, mesh_shape=2,
    float_dtype=jnp.float64, pmid_dtype=jnp.int32, chunk_size=4096,
    lpt_order=2, a_start=0.1, a_stop=1.,
    a_lpt_maxstep=0.05, a_nbody_maxstep=0.1,
)
output_grid = 8
print(f'{conf.ptcl_num} particles; force mesh {conf.mesh_shape}; '
      f'{conf.a_nbody_num} N-body steps; HEFT grid {(output_grid,) * 3}')


The native linear-theory path uses pmwd's Eisenstein–Hu transfer fit and integrated growth, followed by 2LPT and N-body evolution. No external transfer files are needed.

Generate white noise outside any differentiated function. Keep these amplitudes fixed when varying cosmological parameters; a new random seed would change the realization as well as the cosmology.


In [ ]:
cosmo = Cosmology(conf, A_s_1e9=2.1, n_s=0.96, Omega_m=0.3,
                  Omega_b=0.05, h=0.7)
modes = white_noise(17, conf)


## Construct all five fields

The same **unsmoothed** native linear modes feed both gravity and the HEFT branch. Only the HEFT weights receive the Gaussian field-amplitude window $\exp[-k^2/(2k_d^2)]$, with $k_d=0.75\,h\,\mathrm{Mpc}^{-1}$ here. The alternative `smoothing='csst'` uses the radial field window $\exp[-R^2\max(k-k_\mathrm{pivot},0)^2/4]$, with defaults $R=5\,h^{-1}\mathrm{Mpc}$ and $k_\mathrm{pivot}=1\,h\,\mathrm{Mpc}^{-1}$.

Weights refer to `a_ref=1` (redshift zero); evolution ends at `conf.a_stop`. These are separate choices. The reference epoch must lie in the native growth-table range `(0, conf.a_stop]`, so the default reference requires `a_stop >= 1`.


In [ ]:
operators = simulate_heft(
    modes, cosmo, conf, a_ref=1., output_grid=output_grid,
    smoothing='gaussian', kd=0.75,
)
operators = jax.block_until_ready(operators)


Let $A[w]$ be weighted CIC advection with one normalization by output-cell count divided by particle count. The returned channel names are `O1`, `Od`, `OL`, `Od2`, and `Os2`, in the order

$$[A[1]-1,\ A[d],\ A[\nabla^2 d],\ A[Q],\ A[S]],$$

where $d$ is smoothed linear density, $Q=d^2-\langle d^2\rangle_q$, and $S=\sum_{ij}K_{ij}^2-\langle\sum_{ij}K_{ij}^2\rangle_q$ for the traceless tidal tensor $K_{ij}$. Quadratic products use spectral padding and projection, including Nyquist splitting/merging, before advection. No local density division or second trace subtraction is applied.

Output nodes lie at cell centers, offset by half an output cell from the box origin. All channels except the Laplacian are dimensionless; `OL` has units $(h/\mathrm{Mpc})^2$ with the default length unit.


In [ ]:
labels = ['O1: matter overdensity', 'Od: linear density', 'OL: Laplacian',
          'Od2: centered density squared', 'Os2: centered tidal squared']
fields = np.asarray(operators)
assert fields.shape == (5, output_grid, output_grid, output_grid)
assert fields.dtype == np.float64
assert np.isfinite(fields).all()
means = fields.mean(axis=(1, 2, 3))
assert np.max(np.abs(means)) <= 2e-11
print('shape:', fields.shape, 'dtype:', fields.dtype)
for label, mean, rms in zip(labels, means, np.sqrt(np.mean(fields**2, axis=(1, 2, 3)))):
    print(f'{label:29s} mean={mean: .3e}  rms={rms:.3e}')


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(17, 3.5), layout='constrained')
box_size = conf.box_size[0]
for ax, field, label in zip(axes, fields, labels):
    section = field[:, :, output_grid // 2].T
    limit = np.max(np.abs(section))
    picture = ax.imshow(section, origin='lower', extent=(0, box_size, 0, box_size),
                        cmap='RdBu_r', vmin=-limit, vmax=limit)
    ax.set_title(label, fontsize=10)
    ax.set_xlabel(r'$x\ [h^{-1}\mathrm{Mpc}]$')
    fig.colorbar(picture, ax=ax, shrink=0.7)
axes[0].set_ylabel(r'$y\ [h^{-1}\mathrm{Mpc}]$')
plt.show()


## Reuse existing in-memory particles

For an existing simulation, import `heft_operators` and `linear_field_from_modes` from `pmwd.heft`. The lower-level input is a cubic complex rFFT with `norm='forward'` of linear density at the declared reference epoch. Native `linear_modes(..., a=None)` instead uses physical-volume coefficients: the conversion is `native_modes * growth(a_ref) / conf.box_vol`. Growth at `a=1` is **not** assumed to be one. If the modes already contain growth at an epoch, supply `modes_a` to the adapter to avoid applying it twice.

The evolved `Particles` must retain the original C-order Lagrangian row association (z fastest), as native LPT/N-body does. Particle position is `pmid * conf.cell_size + disp`, not displacement alone. Restore row order after any sorting. Low-level field and displacement precision must match `conf.float_dtype`.

The path supports reverse-mode gradients in white-noise amplitudes and continuous cosmological parameters, including both weight and trajectory changes. Keep configuration and HEFT options static when compiling with JAX. Native evolution compiles its own steps; a monolithic JIT is unnecessary. Forward-mode JVP is not promised by native custom-VJP operations.


## Optional final-output save

The result is already available as `operators`. Enable the following cell only to save final fields and their labels. Exclusive creation prevents overwriting an existing result; intermediate arrays are never saved by the HEFT module.


In [ ]:
save_final = False
if save_final:
    from pathlib import Path
    destination = Path('heft_operators.npz')
    with destination.open('xb') as stream:
        np.savez_compressed(stream, operators=fields, labels=np.asarray(labels),
                            a_ref=1., a_stop=conf.a_stop, box_size=conf.box_size[0])
    print(f'Saved {destination}')
